# TFLight model

In [2]:
import lenet
import litert_torch
import litert_quantizer
import model_explorer
import torch

## Load original model

In [3]:
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"
print(f"Using device: {device}")
loaded_model = lenet.LeNetModel().to(device)
loaded_model.load_state_dict(torch.load(lenet.torch_filename))

Using device: cpu


<All keys matched successfully>

## Convert PyTorch to TFLite

In [4]:
with torch.no_grad():
    sample_inputs = (torch.randn(1,1,28,28),)
    edge_model = litert_torch.convert(loaded_model.eval(), sample_inputs)

edge_model.export(lenet.tflight_filename)

(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions

/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


(00:00) [ DONE] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions (+00:00)

(00:00) [ DONE] LiteRT-Torch Convert > Torch Export: serving_default (+00:00)

... (18 outputs left out)

(00:01) [ DONE] LiteRT-Torch Convert (+00:01)

(00:00) [START] Write Model to /home/arykovanov/src/learning/arm_preparation/model/lenet.tflite

I0000 00:00:1794494522.672094 1652380 flatbuffer_export.cc:4410] Estimated count of arithmetic ops: 1.698 M  ops, equivalently 0.849 M  MACs


(00:00) [ DONE] Write Model to /home/arykovanov/src/learning/arm_preparation/model/lenet.tflite (+00:00)

In [41]:
model_explorer.visualize()

Loading extensions...

Loaded 9 adapters:
 - TFLite adapter (Flatbuffer)
 - TFLite adapter (MLIR)
 - TF adapter (MLIR)
 - TF adapter (direct)
 - GraphDef adapter
 - Pytorch adapter (exported program)
 - MLIR adapter
 - HF logic graph adapter (LogicGraph)
 - JSON adapter

Starting Model Explorer server at:
http://localhost:8080

Press Ctrl+C to stop.


I0000 00:00:1794510855.913082 2212264 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Stopping server...


# Run inference on converted model

In [60]:
sample_inputs = lenet.load_tensor(lenet.sample_filename)
image_shape = 1,*sample_inputs[0].shape
image_shape


Tensor shape = (64, 1, 28, 28)


(1, 1, 28, 28)

In [82]:
converted_logits = []
for image in sample_inputs:
    converted_logits.append(edge_model(image.reshape(image_shape))[0])

# Quantize TFLite model

Since our target is running inference on ESP32 with limited resources we need we need to save memory. 
Bacause of this we will use static quantization with quantizing both activations and wrights.

In [83]:
from litert_quantizer import quantizer, recipe
import numpy as np

qt = quantizer.Quantizer(lenet.tflight_filename)
qt.load_quantization_recipe(recipe.static_wi8_ai8())
if qt.need_calibration:
    from litert_quantizer import calibrator
    from ai_edge_litert.interpreter import Interpreter

    samples = lenet.load_tensor(lenet.sample_filename)

    calibration_data = {
        "serving_default": [
            { "args_0": sample[np.newaxis, ...] } for sample in samples
        ]
    }

    calibration_result = qt.calibrate(calibration_data=calibration_data, mode=calibrator.CalibrationMode.CALIBRATION_PROFILER_BASED)

qt.quantize(calibration_result=calibration_result).export_model(lenet.tflight_quantized_filename, overwrite=True)

Tensor shape = (64, 1, 28, 28)


/home/arykovanov/src/learning/arm_preparation/.venv/lib/python3.12/site-packages/ai_edge_litert/interpreter.py:480: UserWarning: Warning: Enabling `experimental_preserve_all_tensors` with the BUILTIN or AUTO op resolver is intended for debugging purposes only. Be aware that this can significantly increase memory usage by storing all intermediate tensors. If you encounter memory problems or are not actively debugging, consider disabling this option.
  warnings.warn(


Model name: /home/arykovanov/src/learning/arm_preparation/model/lenet.tflite
Original model size: 603.66 KiB
Quantized model size: 162.88 KiB
Quantization Ratio: 0.27 (3.7x smaller)
Total time: 8.02 ms


## Compare performance.

## Run quantized inference 

In [68]:
interpreter = Interpreter(model_path=lenet.tflight_quantized_filename)
interpreter.allocate_tensors()
input_info = interpreter.get_input_details()[0]
input_info

{'name': 'serving_default_args_0',
 'index': 0,
 'shape': array([ 1,  1, 28, 28], dtype=int32),
 'shape_signature': array([ 1,  1, 28, 28], dtype=int32),
 'dtype': numpy.int8,
 'quantization': (0.0039205048233270645, -128),
 'quantization_parameters': {'scales': array([0.0039205], dtype=float32),
  'zero_points': array([-128], dtype=int32),
  'quantized_dimension': 0,
  'block_size': 0},
 'sparsity_parameters': {}}

In [75]:
output_info = interpreter.get_output_details()[0]
output_info

{'name': 'serving_default_output_0_output',
 'index': 26,
 'shape': array([ 1, 10], dtype=int32),
 'shape_signature': array([ 1, 10], dtype=int32),
 'dtype': numpy.int8,
 'quantization': (0.07051016390323639, -29),
 'quantization_parameters': {'scales': array([0.07051016], dtype=float32),
  'zero_points': array([-29], dtype=int32),
  'quantized_dimension': 0,
  'block_size': 0},
 'sparsity_parameters': {}}

In [89]:
scale, zero_point = input_info["quantization"]
output_scale, output_zero_point = output_info["quantization"]
quantized_logits = []
for image in sample_inputs:
    image = image.reshape(image_shape)
    quantized_image = np.clip(np.rint(image / scale) + zero_point, -128, 127).astype(np.int8)
    interpreter.set_tensor(input_info["index"], quantized_image)
    interpreter.invoke()

    quantized_output = interpreter.get_tensor(output_info["index"])[0]
    quantized_logits.append((quantized_output.astype(np.float32) - output_zero_point) * output_scale)


In [ ]:
converted_pred = []
quantized_pred = []
for i in range(len(sample_inputs)):
    converted_logit = np.argmax(converted_logits[i])
    quantized_logit = np.argmax(quantized_logits[i])
    converted_pred.append(converted_logit)
    quantized_pred.append(quantized_logit)

    if converted_logit != quantized_logit:
        print(np.round(converted_logits[i], 3))
        print(np.round(quantized_logits[i], 3))
        # print(np.abs(converted_logits[0] - quantized_logits[0]))

        converted_logits_sorted = np.argsort(converted_logits[i], descending=True)
        quantized_logits_sorted = np.argsort(quantized_logits[i], descending=True)
        print(converted_logits_sorted[:2])
        print(quantized_logits_sorted[:2])

        print(f"WARNING: Logits for image {i} mismatch: converted={converted_logit}, quantized={quantized_logit}")



[ 0.263 -3.292  4.982 -0.762  8.241 -5.754  8.355 -7.734  0.509 -4.237]
[ 0.141 -3.243  5.006 -0.776  8.179 -5.57   8.109 -6.981  0.564 -4.09 ]
[6 4]
[4 6]


### Compare accuracy

In [118]:
sample_labels = lenet.load_tensor(lenet.labels_filename, dtype="<i8")

Tensor shape = (64,)


In [125]:
corect_num = sample_labels.shape[0]
converted_correct_num = len(sample_labels[converted_pred == sample_labels])
quantized_correct_num = len(sample_labels[quantized_pred == sample_labels])


print(f"Converted model accuracy: {converted_correct_num/corect_num}")
print(f"Quantized model accuracy: {quantized_correct_num/corect_num}")



Converted model accuracy: 0.828125
Quantized model accuracy: 0.8125
